# Notebook 5: Calibrate Router Handoff Thresholds

Bu notebook router crop/part handoff davranisini offline eval setiyle kalibre eder. Notebook hucresinde yeni calibration mantigi yazmaz; maintained script yuzeylerini calistirir.

Beklenen full-router eval klasor yapisi:

```text
data/router_eval/
  id/<crop>/<part>/*
  negatives/off_crop/<label>/*
  negatives/non_plant/<label>/*
  ambiguous/<label>/*
  wrong_part/<crop>/<unsupported_part>/*
```

Calibration ciktisi router esikleri ve evidence agirliklari icin config override onerir. `config/base.json` otomatik degismez.


## Hizli Akis

Bu notebook router handoff esiklerini offline eval setiyle olcer ve config override onerisi uretir; config dosyasini otomatik degistirmez.

- Genelde sadece `ROUTER_EVAL_ROOT`, `CALIBRATION_PRESET` ve hedef risk limitlerini degistirin.
- Negatif false accept oranini, crop accuracy drop ve part precision drop ile birlikte okuyun.
- Onerilen override dosyasi `CALIBRATION_OUTPUT` altina yazilir; holdout dogrulamasi `HOLDOUT_VALIDATION_OUTPUT` altina yazilir.
- Dev setinde iyi olup `data/router_eval_holdout` uzerinde bozulan ayari kabul etmeyin; config preview holdout kabulunu tercih eder.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell01_bootstrap_access.py', globals())


In [ ]:
import json
from pathlib import Path

from scripts.calibrate_router_surface import calibrate_router_surface, validate_router_candidate_overrides
from scripts.evaluate_router_surface import discover_eval_samples, evaluate_router_surface

# Hizli kullanim: Genelde ROUTER_EVAL_ROOT, CALIBRATION_PRESET ve risk limitlerini degistirin.
CONFIG_ENV = 'colab'
DEVICE = 'cuda'

ROUTER_EVAL_ROOT = 'data/router_eval'
HOLDOUT_EVAL_ROOT = 'data/router_eval_holdout'
BASELINE_EVAL_OUTPUT = '.runtime_tmp/router_eval.json'
CALIBRATION_OUTPUT = '.runtime_tmp/router_calibration.json'
HOLDOUT_VALIDATION_OUTPUT = '.runtime_tmp/router_calibration_holdout.json'

RUN_BASELINE_EVAL = True
RUN_CALIBRATION = True
RUN_HOLDOUT_VALIDATION = True

CALIBRATION_PRESET = 'quick'  # none, handoff, quick, docs
CUSTOM_SWEEPS = []
# Examples:
# CUSTOM_SWEEPS = ['router_min_confidence=0.55,0.65,0.75', 'router_min_margin=0.0,0.1,0.15']

MAX_VARIANTS = 128
TARGET_NEGATIVE_FALSE_ACCEPT_RATE = 0.05
MAX_CROP_ACCURACY_DROP = 0.02
MAX_PART_PRECISION_DROP = 0.02
MAX_PART_RECALL_DROP = 0.02
MAX_WRONG_PART_REJECTION_DROP = 0.02
MAX_P95_LATENCY_REGRESSION = 0.25
HOLDOUT_TOP_K = 3
INCLUDE_SAMPLES_IN_OUTPUT = False

def _write_json(path_value, payload):
    path = Path(path_value)
    if str(path):
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        print(f'[OUTPUT] wrote {path}')

root = Path(ROUTER_EVAL_ROOT)
print(f'[CONFIG] eval_root={root}')
print(f'[CONFIG] config_env={CONFIG_ENV} device={DEVICE} preset={CALIBRATION_PRESET}')
print(
    f'[RISK_TARGETS] negative_fa<={TARGET_NEGATIVE_FALSE_ACCEPT_RATE:.3f} '
    f'crop_drop<={MAX_CROP_ACCURACY_DROP:.3f} part_precision_drop<={MAX_PART_PRECISION_DROP:.3f} '
    f'part_recall_drop<={MAX_PART_RECALL_DROP:.3f} wrong_part_drop<={MAX_WRONG_PART_REJECTION_DROP:.3f}'
)
print(f'[OUTPUT] baseline={BASELINE_EVAL_OUTPUT} calibration={CALIBRATION_OUTPUT} holdout={HOLDOUT_VALIDATION_OUTPUT}')
print(f'[HOLDOUT] eval_root={HOLDOUT_EVAL_ROOT} top_k={HOLDOUT_TOP_K}')
print('[SONRAKI] Data summary -> baseline eval -> calibration -> holdout validation -> config preview hucresini sirayla calistirin.')


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell03_data_summary.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell04_baseline_eval.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell05_calibration.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell06_holdout_validation.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb5_cell06_config_preview.py', globals())
